In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os, pickle
import numpy as np
import pandas as pd


for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import os, pickle, numpy as np, pandas as pd

# --------------------- Utilitare I/O ---------------------
TRAIN_PKL = "/kaggle/input/fii-nn-2025-homework-2/extended_mnist_train.pkl"
TEST_PKL  = "/kaggle/input/fii-nn-2025-homework-2/extended_mnist_test.pkl"
if not os.path.exists(TRAIN_PKL): TRAIN_PKL = "extended_mnist_train.pkl"
if not os.path.exists(TEST_PKL):  TEST_PKL  = "extended_mnist_test.pkl"

def _load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def _to_numpy_flat(images):
    X = [np.asarray(img).reshape(-1) for img in images]
    X = np.stack(X).astype("float32")
    if X.max() > 1.0:
        X /= 255.0
    return X

# --------------------- softmax  ---------------------
def softMax(Z):
    goodZ = Z - np.max(Z, axis=1, keepdims=True)
    expZ  = np.exp(goodZ)
    return expZ / np.sum(expZ, axis=1, keepdims=True)

def one_hot(y, num_classes):
    return np.eye(num_classes, dtype=np.float32)[y]

def accuracy(y_true_oh, y_pred_oh):
    return np.mean(np.argmax(y_true_oh, axis=1) == np.argmax(y_pred_oh, axis=1))

def iterate_batches(X, y, batch_size, shuffle=True):
    n = X.shape[0]
    idx = np.arange(n)
    if shuffle: np.random.shuffle(idx)
    for i in range(0, n, batch_size):
        sel = idx[i:i+batch_size]
        yield X[sel], y[sel]

def forward(X, W, b):
    return softMax(X @ W + b)

def cross_entropy(y_true, y_pred):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-12), axis=1))

def backpropagation(X, y_true, y_pred):
    m  = X.shape[0]
    dZ = y_pred - y_true
    dW = (X.T @ dZ) / m
    db = np.sum(dZ, axis=0) / m
    return dW, db

def update(W, b, dW, db, learning_rate):
    W -= learning_rate * dW
    b -= learning_rate * db
    return W, b

def train_softmax_numpy(
    X_train, y_train, X_val=None, y_val=None,
    num_classes=None, epochs=50, batch_size=100, learning_rate=0.1, seed=42
):
    if num_classes is None:
        num_classes = int(np.max(y_train)) + 1
    y_train_oh = one_hot(y_train, num_classes)
    y_val_oh   = one_hot(y_val, num_classes) if y_val is not None else None

    np.random.seed(seed)
    n_features = X_train.shape[1]
    W = (np.random.randn(n_features, num_classes).astype(np.float32)) * 0.01
    b = np.zeros((num_classes,), dtype=np.float32)

    # Acuratete initiala
    y_pred_init = forward(X_train, W, b)
    init_acc = accuracy(y_train_oh, y_pred_init)
    print(f"Initial train accuracy (untrained): {init_acc*100:.2f}%")

    for epoch in range(1, epochs + 1):
        # antrenare pe mini-batch
        for Xb, yb_oh in iterate_batches(X_train, y_train_oh, batch_size, shuffle=True):
            yb_pred = forward(Xb, W, b)
            dW, db  = backpropagation(Xb, yb_oh, yb_pred)
            W, b    = update(W, b, dW, db, learning_rate)

        # metrici
        y_pred_train = forward(X_train, W, b)
        train_loss = cross_entropy(y_train_oh, y_pred_train)
        train_acc  = accuracy(y_train_oh, y_pred_train)

        if y_val_oh is not None:
            y_pred_val = forward(X_val, W, b)
            val_acc = accuracy(y_val_oh, y_pred_val)
            print(f"Epoch {epoch:3d} | loss={train_loss:.4f} | train_acc={train_acc*100:5.2f}% | val_acc={val_acc*100:5.2f}%")
        else:
            print(f"Epoch {epoch:3d} | loss={train_loss:.4f} | train_acc={train_acc*100:5.2f}%")

    return W, b

# --------------------- 1) Date pickle ---------------------
train_list = _load_pickle(TRAIN_PKL)  # lista de (img,eticheta)
test_list  = _load_pickle(TEST_PKL)   

train_imgs, train_labels = [], []
for img, lab in train_list:
    train_imgs.append(img)
    train_labels.append(int(lab))

test_imgs = []
for it in test_list:
    if isinstance(it, (list, tuple)) and len(it) >= 1:
        test_imgs.append(it[0])
    else:
        test_imgs.append(it)

X = _to_numpy_flat(train_imgs)
y = np.asarray(train_labels, dtype=np.int64)
X_test = _to_numpy_flat(test_imgs)

print(f"X_train={X.shape}, y_train={y.shape}, X_test={X_test.shape}")

# --------------------- 2) Split simplu train/val (10%) ---------------------
n = X.shape[0]
perm = np.random.RandomState(42).permutation(n)
val_count = max(1, int(0.1 * n))
val_idx, tr_idx = perm[:val_count], perm[val_count:]
X_tr, y_tr = X[tr_idx], y[tr_idx]
X_val, y_val = X[val_idx], y[val_idx]

# --------------------- 3) Antrenare  ---------------------
W, b = train_softmax_numpy(
    X_train=X_tr, y_train=y_tr,
    X_val=X_val,   y_val=y_val,
    num_classes=int(np.max(y)) + 1,
    epochs=120,          
    batch_size=100,      
    learning_rate=0.2,   
    seed=42,
)

# --------------------- 4) Predictii pe test ---------------------
probs_test = forward(X_test, W, b)
y_pred = np.argmax(probs_test, axis=1).astype(int)

# --------------------- 5) Submission ---------------------
submission = pd.DataFrame({"ID": np.arange(len(y_pred)), "target": y_pred})
out_path = "/kaggle/working/submission.csv" if os.path.exists("/kaggle/working") else "submission.csv"
submission.to_csv(out_path, index=False)
print(f"\nFișierul a fost salvat la: {out_path}")
display(submission.head(10))